### Apriori Algorithm
This algorithms calculates the support, confidence and lift for all possible combination of products which satisfies a minimium threshold of support and confidence. Hence reducing the computation required. Otherwise there will be too many combinations.

**This activity is performed in two major steps**
* Find out the frequent combination of items called "itemsets" which satisfy the minimium support threshold.
* Generate association from frequent itemsets rules based on the minimium cconfidence threshold.

The above style of finding association rule is based on breadth-first search strategy, hence it needs to scan the database multiple times. Thats why apriori algorithm is slow. it is often criticized for not being able to handle large datasets.

In [1]:
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

The Apriori algorithm is based on three key concepts:
1. Support
2. Confidence
3. Lift

* **Support** refers to the default popularity of an item and can be calculated by finding the number of transactions containing a particular item divided by the total number of transactions. 
* **Confidence** refers to the likelihood that an item B is also bought if item A is bought. It can be calculated by finding the number of transactions where A and B are bought together, divided by the total number of transactions where A is bought.
* **Lift** refers to the increase in the ratio of sale of a product B when another product A is sold. 


The Apriori algorithm uses frequent itemset to generate association rule. Frequent itemsets is those whos support values is greater than threshold values.

#### Data Preparation

In [2]:
trend = pd.read_excel("../data/Online Retail.xlsx")
trend = trend.dropna()
trend.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


For large sets of data, there can be hundreds of items in hundreds of thousands of transactions. The Apriori algorithm tries to extract rules for each possible combination of items. We can set minimum levels of support, confidence or lift to be reached to count something as a “good rule”. 

In [3]:
trend.shape

(406829, 8)

It is too much data, remove some of them by only storing data of france country

In [4]:
data = trend[trend["Country"] == "France"]
data.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
26,536370,22728,ALARM CLOCK BAKELIKE PINK,24,2010-12-01 08:45:00,3.75,12583.0,France
27,536370,22727,ALARM CLOCK BAKELIKE RED,24,2010-12-01 08:45:00,3.75,12583.0,France
28,536370,22726,ALARM CLOCK BAKELIKE GREEN,12,2010-12-01 08:45:00,3.75,12583.0,France
29,536370,21724,PANDA AND BUNNIES STICKER SHEET,12,2010-12-01 08:45:00,0.85,12583.0,France
30,536370,21883,STARS GIFT TAPE,24,2010-12-01 08:45:00,0.65,12583.0,France


In [5]:
data.shape

(8491, 8)

Clean data

In [6]:
data["Description"] = data["Description"].str.strip()

C:\Users\Sakhi Larik\AppData\Local\Temp\ipykernel_1680\2773864452.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["Description"] = data["Description"].str.strip()


In [7]:
data.dropna(axis=0,subset=["InvoiceNo"],inplace=True) # remove duplicate invoice number
data["InvoiceNo"] = data["InvoiceNo"].astype('str') # convert invoice number to string
data = data[~data["InvoiceNo"].str.contains("c")] # remove the credit transaction
data.head()

C:\Users\Sakhi Larik\AppData\Local\Temp\ipykernel_1680\3150730308.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data.dropna(axis=0,subset=["InvoiceNo"],inplace=True) # remove duplicate invoice number
C:\Users\Sakhi Larik\AppData\Local\Temp\ipykernel_1680\3150730308.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["InvoiceNo"] = data["InvoiceNo"].astype('str') # convert invoice number to string


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
26,536370,22728,ALARM CLOCK BAKELIKE PINK,24,2010-12-01 08:45:00,3.75,12583.0,France
27,536370,22727,ALARM CLOCK BAKELIKE RED,24,2010-12-01 08:45:00,3.75,12583.0,France
28,536370,22726,ALARM CLOCK BAKELIKE GREEN,12,2010-12-01 08:45:00,3.75,12583.0,France
29,536370,21724,PANDA AND BUNNIES STICKER SHEET,12,2010-12-01 08:45:00,0.85,12583.0,France
30,536370,21883,STARS GIFT TAPE,24,2010-12-01 08:45:00,0.65,12583.0,France


##### Convert Data From Row To Columns and number of purchase
Group items with invoice no to find which item is also purchased along with that by the same customer. This helps to find the confidence of the item

In [8]:
basket = (data.groupby(["InvoiceNo","Description"])["Quantity"]
          .sum().unstack().reset_index().fillna(0)
          .set_index("InvoiceNo"))

In [9]:
basket.head()

Description,10 COLOUR SPACEBOY PEN,12 COLOURED PARTY BALLOONS,12 EGG HOUSE PAINTED WOOD,12 MESSAGE CARDS WITH ENVELOPES,12 PENCIL SMALL TUBE WOODLAND,12 PENCILS SMALL TUBE RED RETROSPOT,12 PENCILS SMALL TUBE SKULL,12 PENCILS TALL TUBE POSY,12 PENCILS TALL TUBE RED RETROSPOT,12 PENCILS TALL TUBE WOODLAND,...,WRAP SUKI AND FRIENDS,WRAP VINTAGE PETALS DESIGN,YELLOW COAT RACK PARIS FASHION,YELLOW GIANT GARDEN THERMOMETER,ZINC STAR T-LIGHT HOLDER,ZINC FOLKART SLEIGH BELLS,ZINC HERB GARDEN CONTAINER,ZINC METAL HEART DECORATION,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS SMALL
InvoiceNo,,,,,,,,,,,,,,,,,,,,,
536370,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
536852,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
536974,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
537065,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
537463,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


#### Hot encoding
to convert large number than 0 to 1 and other number to 1 only

#### Now we neet to do hot encode to convert values into 0 and 1 only

In [10]:
def hot_encode(x):
    if(x<=0):
        return 0
    if(x>=1):
        return 1

In [11]:
basket_set = basket.applymap(hot_encode)
basket_set.drop('POSTAGE',inplace=True,axis=1)

C:\Users\Sakhi Larik\AppData\Local\Temp\ipykernel_1680\84246596.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  basket_set = basket.applymap(hot_encode)


In [12]:
basket_set.head()

Description,10 COLOUR SPACEBOY PEN,12 COLOURED PARTY BALLOONS,12 EGG HOUSE PAINTED WOOD,12 MESSAGE CARDS WITH ENVELOPES,12 PENCIL SMALL TUBE WOODLAND,12 PENCILS SMALL TUBE RED RETROSPOT,12 PENCILS SMALL TUBE SKULL,12 PENCILS TALL TUBE POSY,12 PENCILS TALL TUBE RED RETROSPOT,12 PENCILS TALL TUBE WOODLAND,...,WRAP SUKI AND FRIENDS,WRAP VINTAGE PETALS DESIGN,YELLOW COAT RACK PARIS FASHION,YELLOW GIANT GARDEN THERMOMETER,ZINC STAR T-LIGHT HOLDER,ZINC FOLKART SLEIGH BELLS,ZINC HERB GARDEN CONTAINER,ZINC METAL HEART DECORATION,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS SMALL
InvoiceNo,,,,,,,,,,,,,,,,,,,,,
536370,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536852,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536974,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
537065,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
537463,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [21]:
basket.shape

(458, 1544)

### Do Machine Learning (Training the Model)

#### Generate Frequent Itemset

In [13]:
freq_item = apriori(basket_set,min_support=0.07,use_colnames=True)

c:\Users\Sakhi Larik\AppData\Local\Programs\Python\Python311\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:110: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


In [14]:
freq_item.head()

,support,itemsets
0,0.082969,(ALARM CLOCK BAKELIKE GREEN)
1,0.087336,(ALARM CLOCK BAKELIKE PINK)
2,0.080786,(ALARM CLOCK BAKELIKE RED)
3,0.085153,(DOLLY GIRL LUNCH BOX)
4,0.082969,(JUMBO BAG RED RETROSPOT)


#### Apply Rules

In [15]:
rules = association_rules(freq_item,metric="lift",min_threshold=1)

In [16]:
rules.head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
0,(PLASTERS IN TIN CIRCUS PARADE),(PLASTERS IN TIN SPACEBOY),0.144105,0.115721,0.076419,0.530303,4.582619,0.059743,1.882660,0.913411
1,(PLASTERS IN TIN SPACEBOY),(PLASTERS IN TIN CIRCUS PARADE),0.115721,0.144105,0.076419,0.660377,4.582619,0.059743,2.520136,0.884092
2,(PLASTERS IN TIN WOODLAND ANIMALS),(PLASTERS IN TIN CIRCUS PARADE),0.146288,0.144105,0.087336,0.597015,4.142922,0.066255,2.123888,0.888619
3,(PLASTERS IN TIN CIRCUS PARADE),(PLASTERS IN TIN WOODLAND ANIMALS),0.144105,0.146288,0.087336,0.606061,4.142922,0.066255,2.167115,0.886352
4,(PLASTERS IN TIN WOODLAND ANIMALS),(PLASTERS IN TIN SPACEBOY),0.146288,0.115721,0.089520,0.611940,5.288088,0.072591,2.278720,0.949847


In [17]:
rules.shape

(18, 10)

#### Making Recomendations

In [18]:
# Sum of PLASTERS IN TIN CIRCUS PARADE in the basket is
basket_set['PLASTERS IN TIN CIRCUS PARADE'].sum()

66

In [19]:
# Sum of PLASTERS IN TIN SPACEBOY in the basket is
basket_set['PLASTERS IN TIN SPACEBOY'].sum()

53

In [20]:
# Filtering rules based on condition
rules[(rules['lift']>=3) & (rules['confidence'] >=0.75)]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
5,(PLASTERS IN TIN SPACEBOY),(PLASTERS IN TIN WOODLAND ANIMALS),0.115721,0.146288,0.089520,0.773585,5.288088,0.072591,3.770560,0.917013
7,(SET/20 RED RETROSPOT PAPER NAPKINS),(SET/6 RED SPOTTY PAPER CUPS),0.113537,0.117904,0.087336,0.769231,6.524217,0.073950,3.822416,0.955172
8,(SET/20 RED RETROSPOT PAPER NAPKINS),(SET/6 RED SPOTTY PAPER PLATES),0.113537,0.109170,0.087336,0.769231,7.046154,0.074941,3.860262,0.967980
9,(SET/6 RED SPOTTY PAPER PLATES),(SET/20 RED RETROSPOT PAPER NAPKINS),0.109170,0.113537,0.087336,0.800000,7.046154,0.074941,4.432314,0.963235
10,(SET/6 RED SPOTTY PAPER CUPS),(SET/6 RED SPOTTY PAPER PLATES),0.117904,0.109170,0.104803,0.888889,8.142222,0.091932,8.017467,0.994431
11,(SET/6 RED SPOTTY PAPER PLATES),(SET/6 RED SPOTTY PAPER CUPS),0.109170,0.117904,0.104803,0.960000,8.142222,0.091932,22.052402,0.984681
12,"(SET/6 RED SPOTTY PAPER CUPS, SET/20 RED RETRO...",(SET/6 RED SPOTTY PAPER PLATES),0.087336,0.109170,0.085153,0.975000,8.931000,0.075618,35.633188,0.973009
13,"(SET/6 RED SPOTTY PAPER CUPS, SET/6 RED SPOTTY...",(SET/20 RED RETROSPOT PAPER NAPKINS),0.104803,0.113537,0.085153,0.812500,7.156250,0.073254,4.727802,0.960976
14,"(SET/20 RED RETROSPOT PAPER NAPKINS, SET/6 RED...",(SET/6 RED SPOTTY PAPER CUPS),0.087336,0.117904,0.085153,0.975000,8.269444,0.074856,35.283843,0.963195
16,(SET/20 RED RETROSPOT PAPER NAPKINS),"(SET/6 RED SPOTTY PAPER CUPS, SET/6 RED SPOTTY...",0.113537,0.104803,0.085153,0.750000,7.156250,0.073254,3.580786,0.970443
